In [8]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv
import graphviz
import json
from graphviz import Digraph

In [9]:
load_dotenv()

True

In [10]:
llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [11]:
text = """
In 2017, a young engineer named Arjun Mehta started working at Tesla as a battery research intern. During his internship, he contributed to the development of 4680 battery cells, which later improved the Model Y’s range by 15%.
In 2019, after graduating from IIT Bombay, Arjun joined Tesla full-time as a Battery Systems Engineer. He collaborated closely with Dr. Lisa Wong, a materials scientist, to optimize the anode chemistry using silicon nanowires.
Their research paper titled “Silicon Nanowire Anodes for High-Energy Batteries” was published in Nature Energy in 2020, gaining widespread recognition.
In 2021, Arjun moved to SpaceX to work on Starship's thermal protection systems, applying his materials expertise.
By 2023, he founded his own startup NanoVolt Energy, aiming to commercialize solid-state lithium batteries for electric aviation.
Today, NanoVolt has raised $50 million in Series A funding led by Sequoia Capital and is working with companies like Joby Aviation to integrate these batteries into their eVTOL aircraft.
"""

In [12]:
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["text"],
    template="""
You are an advanced Knowledge Graph Extraction AI.

Your task is to read the given text and extract all entities and their relationships in the form of structured JSON triples. Each triple should have:

- subject: The main entity or actor.
- relation: The relationship or action connecting subject and object.
- object: The target entity or item acted upon.

### Instructions:
- Capture people, organizations, dates, events, technologies, products, etc.
- Keep relation names concise, using verbs like "worked at", "founded", "published in", "collaborated with", "developed", "improved", "graduated from", "raised", "partnered with".
- If an entity is associated with a date or year, include it as a separate triple with relation “date” or integrate it into the event triple if contextually meaningful.

### Output formatting rules:
- Return ONLY a valid JSON array of triples.
- Do NOT include markdown formatting (no ```json or ```).
- Do NOT include any explanation text, comments, or additional messages.
- Ensure the JSON is parsable directly without further cleaning.

### Text to process:

{text}
"""
)

final_prompt = prompt_template.format(text=text)


In [13]:
response = llm.invoke(final_prompt)
print(response)

content='[\n    {"subject": "Arjun Mehta", "relation": "started working at", "object": "Tesla", "date": "2017"},\n    {"subject": "Arjun Mehta", "relation": "contributed to", "object": "development of 4680 battery cells"},\n    {"subject": "4680 battery cells", "relation": "improved", "object": "Model Y’s range"},\n    {"subject": "Arjun Mehta", "relation": "graduated from", "object": "IIT Bombay", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "joined as", "object": "Battery Systems Engineer at Tesla", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "collaborated with", "object": "Dr. Lisa Wong"},\n    {"subject": "Dr. Lisa Wong", "relation": "collaborated with", "object": "Arjun Mehta"},\n    {"subject": "Arjun Mehta & Dr. Lisa Wong", "relation": "optimized", "object": "anode chemistry using silicon nanowires"},\n    {"subject": "research paper", "relation": "titled", "object": "Silicon Nanowire Anodes for High-Energy Batteries"},\n    {"subject": "research

In [14]:
type(response.content)

str

In [15]:
response.content

'[\n    {"subject": "Arjun Mehta", "relation": "started working at", "object": "Tesla", "date": "2017"},\n    {"subject": "Arjun Mehta", "relation": "contributed to", "object": "development of 4680 battery cells"},\n    {"subject": "4680 battery cells", "relation": "improved", "object": "Model Y’s range"},\n    {"subject": "Arjun Mehta", "relation": "graduated from", "object": "IIT Bombay", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "joined as", "object": "Battery Systems Engineer at Tesla", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "collaborated with", "object": "Dr. Lisa Wong"},\n    {"subject": "Dr. Lisa Wong", "relation": "collaborated with", "object": "Arjun Mehta"},\n    {"subject": "Arjun Mehta & Dr. Lisa Wong", "relation": "optimized", "object": "anode chemistry using silicon nanowires"},\n    {"subject": "research paper", "relation": "titled", "object": "Silicon Nanowire Anodes for High-Energy Batteries"},\n    {"subject": "research paper",

In [16]:
json.loads(response.content)

[{'subject': 'Arjun Mehta',
  'relation': 'started working at',
  'object': 'Tesla',
  'date': '2017'},
 {'subject': 'Arjun Mehta',
  'relation': 'contributed to',
  'object': 'development of 4680 battery cells'},
 {'subject': '4680 battery cells',
  'relation': 'improved',
  'object': 'Model Y’s range'},
 {'subject': 'Arjun Mehta',
  'relation': 'graduated from',
  'object': 'IIT Bombay',
  'date': '2019'},
 {'subject': 'Arjun Mehta',
  'relation': 'joined as',
  'object': 'Battery Systems Engineer at Tesla',
  'date': '2019'},
 {'subject': 'Arjun Mehta',
  'relation': 'collaborated with',
  'object': 'Dr. Lisa Wong'},
 {'subject': 'Dr. Lisa Wong',
  'relation': 'collaborated with',
  'object': 'Arjun Mehta'},
 {'subject': 'Arjun Mehta & Dr. Lisa Wong',
  'relation': 'optimized',
  'object': 'anode chemistry using silicon nanowires'},
 {'subject': 'research paper',
  'relation': 'titled',
  'object': 'Silicon Nanowire Anodes for High-Energy Batteries'},
 {'subject': 'research paper',


In [17]:
kg_str = response.content

In [18]:

# Parse string to JSON
kg = json.loads(kg_str)
# Initialize Graphviz directed graph
dot = Digraph(comment='Knowledge Graph', format='png')
dot.attr('node', shape='ellipse')

# Add edges from triples
for triple in kg:
    subj = triple['subject']
    obj = triple['object']
    rel = triple['relation']
    
    label = rel
    # If 'date' field exists, append to relation label
    if 'date' in triple:
        label += f" ({triple['date']})"
    
    dot.edge(subj, obj, label=label)

# Render and open the graph
dot.render('knowledge_graph', view=True)


'knowledge_graph.png'

In [19]:
timeline_prompt = f"""
You are an expert timeline extraction AI.

Given the following text, extract a **chronological timeline** as a JSON array where each element has:

- "year": Year of the event (if available)
- "event": Short description of what happened

Return only the JSON array, no explanations or markdown.

Text:
{text}
"""


In [20]:
timeline_from_kg_prompt = f"""
You are an expert timeline generation AI.

Using the following knowledge graph triples, generate a chronological timeline in JSON format. Each entry should include:

- "year": Year of the event (if available)
- "event": Short description of what happened

Knowledge Graph Triples:
{kg_str}

Return only the JSON array, no explanations or markdown.
"""


In [21]:
respone_without_kg = llm.invoke(timeline_prompt)

In [22]:
print(respone_without_kg)

content='[\n    {\n        "year": 2017,\n        "event": "Arjun Mehta started working at Tesla as a battery research intern"\n    },\n    {\n        "year": 2019,\n        "event": "Arjun joined Tesla full-time as a Battery Systems Engineer"\n    },\n    {\n        "year": 2020,\n        "event": "Research paper titled \'Silicon Nanowire Anodes for High-Energy Batteries\' published in Nature Energy"\n    },\n    {\n        "year": 2021,\n        "event": "Arjun moved to SpaceX to work on Starship\'s thermal protection systems"\n    },\n    {\n        "year": 2023,\n        "event": "Arjun founded NanoVolt Energy to commercialize solid-state lithium batteries for electric aviation"\n    }\n]' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 296, 'total_tokens': 464, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_det

In [23]:
json.loads(respone_without_kg.content)

[{'year': 2017,
  'event': 'Arjun Mehta started working at Tesla as a battery research intern'},
 {'year': 2019,
  'event': 'Arjun joined Tesla full-time as a Battery Systems Engineer'},
 {'year': 2020,
  'event': "Research paper titled 'Silicon Nanowire Anodes for High-Energy Batteries' published in Nature Energy"},
 {'year': 2021,
  'event': "Arjun moved to SpaceX to work on Starship's thermal protection systems"},
 {'year': 2023,
  'event': 'Arjun founded NanoVolt Energy to commercialize solid-state lithium batteries for electric aviation'}]

In [24]:
timeline_from_kg_prompt

'\nYou are an expert timeline generation AI.\n\nUsing the following knowledge graph triples, generate a chronological timeline in JSON format. Each entry should include:\n\n- "year": Year of the event (if available)\n- "event": Short description of what happened\n\nKnowledge Graph Triples:\n[\n    {"subject": "Arjun Mehta", "relation": "started working at", "object": "Tesla", "date": "2017"},\n    {"subject": "Arjun Mehta", "relation": "contributed to", "object": "development of 4680 battery cells"},\n    {"subject": "4680 battery cells", "relation": "improved", "object": "Model Y’s range"},\n    {"subject": "Arjun Mehta", "relation": "graduated from", "object": "IIT Bombay", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "joined as", "object": "Battery Systems Engineer at Tesla", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "collaborated with", "object": "Dr. Lisa Wong"},\n    {"subject": "Dr. Lisa Wong", "relation": "collaborated with", "object": "Arjun 

In [25]:
response_with_kg = llm.invoke(timeline_from_kg_prompt)

In [26]:
response_with_kg

AIMessage(content='[\n    {"year": "2017", "event": "Arjun Mehta started working at Tesla"},\n    {"event": "Arjun Mehta contributed to development of 4680 battery cells"},\n    {"event": "4680 battery cells improved Model Y’s range"},\n    {"year": "2019", "event": "Arjun Mehta graduated from IIT Bombay"},\n    {"event": "Arjun Mehta joined as Battery Systems Engineer at Tesla"},\n    {"event": "Arjun Mehta collaborated with Dr. Lisa Wong"},\n    {"event": "Arjun Mehta & Dr. Lisa Wong optimized anode chemistry using silicon nanowires"},\n    {"year": "2020", "event": "Research paper titled Silicon Nanowire Anodes for High-Energy Batteries published in Nature Energy"},\n    {"year": "2021", "event": "Arjun Mehta moved to SpaceX"},\n    {"event": "Arjun Mehta works on Starship\'s thermal protection systems at SpaceX"},\n    {"year": "2023", "event": "Arjun Mehta founded NanoVolt Energy"},\n    {"event": "NanoVolt Energy aims to commercialize solid-state lithium batteries for electric av

In [27]:
timeline = json.loads(response_with_kg.content)

In [28]:
timeline

[{'year': '2017', 'event': 'Arjun Mehta started working at Tesla'},
 {'event': 'Arjun Mehta contributed to development of 4680 battery cells'},
 {'event': '4680 battery cells improved Model Y’s range'},
 {'year': '2019', 'event': 'Arjun Mehta graduated from IIT Bombay'},
 {'event': 'Arjun Mehta joined as Battery Systems Engineer at Tesla'},
 {'event': 'Arjun Mehta collaborated with Dr. Lisa Wong'},
 {'event': 'Arjun Mehta & Dr. Lisa Wong optimized anode chemistry using silicon nanowires'},
 {'year': '2020',
  'event': 'Research paper titled Silicon Nanowire Anodes for High-Energy Batteries published in Nature Energy'},
 {'year': '2021', 'event': 'Arjun Mehta moved to SpaceX'},
 {'event': "Arjun Mehta works on Starship's thermal protection systems at SpaceX"},
 {'year': '2023', 'event': 'Arjun Mehta founded NanoVolt Energy'},
 {'event': 'NanoVolt Energy aims to commercialize solid-state lithium batteries for electric aviation'},
 {'event': 'NanoVolt raised $50 million in Series A fundin

In [29]:
import textwrap
import math

dot = Digraph(comment='Career Timeline', format='png')
dot.attr(rankdir='TB', size='10,10', dpi='300')  # Top to bottom layout for rows
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightgrey')

# Parameters
chunk_size = 5
num_chunks = math.ceil(len(timeline) / chunk_size)

prev_chunk_last_node = None

for chunk_idx in range(num_chunks):
    with dot.subgraph() as s:
        s.attr(rank='same')  # Same rank for horizontal alignment

        prev_node = None
        for i in range(chunk_idx * chunk_size, min((chunk_idx + 1) * chunk_size, len(timeline))):
            entry = timeline[i]
            label = ''
            if 'year' in entry:
                label += f"{entry['year']}: "
            wrapped_event = '\n'.join(textwrap.wrap(entry['event'], width=30))
            label += wrapped_event

            node_id = f"e{i}"
            s.node(node_id, label)

            if prev_node:
                s.edge(prev_node, node_id)

            prev_node = node_id

        # Connect previous chunk to current chunk's first node for sequential flow
        if prev_chunk_last_node and (chunk_idx * chunk_size < len(timeline)):
            dot.edge(prev_chunk_last_node, f"e{chunk_idx * chunk_size}")

        prev_chunk_last_node = prev_node

dot.render('timeline_split_graph', view=True)

'timeline_split_graph.png'

In [30]:
from graphviz import Digraph

dot = Digraph(comment='Knowledge Graph Example', format='png')
dot.node('A', 'Arjun Mehta')
dot.node('B', 'Tesla')
dot.edge('A', 'B', label='worked at')

dot.render('knowledge_graph_example', view=True)

'knowledge_graph_example.png'

In [31]:
import textwrap
import math

dot = Digraph(comment='Career Timeline', format='png')
dot.attr(rankdir='TB', size='10,10', dpi='300')  # Top to bottom layout for rows
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightgrey')

# Parameters
chunk_size = 5
num_chunks = math.ceil(len(timeline) / chunk_size)

prev_chunk_last_node = None

for chunk_idx in range(num_chunks):
    with dot.subgraph() as s:
        s.attr(rank='same')  # Same rank for horizontal alignment

        prev_node = None
        for i in range(chunk_idx * chunk_size, min((chunk_idx + 1) * chunk_size, len(timeline))):
            entry = timeline[i]
            label = ''
            if 'year' in entry:
                label += f"{entry['year']}: "
            wrapped_event = '\n'.join(textwrap.wrap(entry['event'], width=30))
            label += wrapped_event

            node_id = f"e{i}"
            s.node(node_id, label)

            if prev_node:
                s.edge(prev_node, node_id)

            prev_node = node_id

        # Connect previous chunk to current chunk's first node for sequential flow
        if prev_chunk_last_node and (chunk_idx * chunk_size < len(timeline)):
            dot.edge(prev_chunk_last_node, f"e{chunk_idx * chunk_size}")

        prev_chunk_last_node = prev_node

dot.render('timeline_split_graph', view=True)

'timeline_split_graph.png'